# Surface water-mass transformation

Surface water-mass transformation south of 60S, after Newsom et
al. (2016) and Abernathey et al. (2016): the volume flux into a
density class driven by surface buoyancy forcing, split into its
heat and salt contributions.

$$\Omega(\sigma) = \frac{\partial}{\partial\sigma}
\iint_{\sigma' < \sigma}
\left(\frac{\partial\sigma}{\partial\theta}\dot\theta
+ \frac{\partial\sigma}{\partial S}\dot S\right)
\mathrm{d}x\,\mathrm{d}y$$

The notebook's MOM6 `salt_flux` maps onto CMIP6 `sfdsi`, which is
the sea-ice salt flux only. Models that also restore surface
salinity account for that elsewhere, so where restoring matters
the salt contribution here is a lower bound.

The monthly time axis must survive the preprocessor: the binning
needs the density of each cell *in each month*, not of the
climatology.

---

Translated from the COSIMA recipe
[`Surface_Water_Mass_Transformation.ipynb`](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Surface_Water_Mass_Transformation.ipynb)
onto CMORised CMIP6 data, using ESMValCore's `Dataset` and
preprocessors in place of the ACCESS-NRI intake catalog.

This notebook is standalone: it needs only `esmvalcore`,
`iris`, `numpy`, `matplotlib`, `cartopy` and `gsw`. The
equivalent ESMValTool recipe is
`recipes/recipe_cosima_*.yml`.

In [ ]:
import numpy as np
import iris
import iris.analysis.cartography
import iris.plot as iplt
import matplotlib.pyplot as plt

from esmvalcore.dataset import Dataset
import gsw
from esmvalcore.preprocessor import extract_region, regrid

## Start a dask cluster

Every COSIMA recipe opens with one, and these need one for the same
reason, though it is worth being precise about *which* part it speeds up.

**What the cluster does accelerate.** Everything from `Dataset.load()`
through the ESMValCore preprocessors — reading multi-decade files,
`regrid`, `climate_statistics`, `extract_region` — is lazy and chunked.
That is the expensive part of these notebooks and it is what the workers
parallelise. Without a client, dask falls back to its threaded scheduler;
ESMValCore itself warns that the distributed scheduler is preferable.

**What it does not.** Once the preprocessed cube is reduced, the
diagnostic arithmetic realises it into a plain masked array and proceeds
eagerly in numpy. That is deliberate: contour tracing, per-column
remapping and density binning are iterative and index-based, so they do
not express well as array graphs and would be slower, not faster, spread
over workers. Realising *after* the time-mean is what keeps that
affordable — realising a 20-year daily 3-D field before reducing it would
not be.

So: size the cluster for the load-and-preprocess stage. One thread per
worker, as the COSIMA recipes use — with chunks this large, per-worker
memory binds well before thread count does.

On ARE, `Client(threads_per_worker=1)` picks up the cores you requested
for the session. Pass `n_workers=` and `memory_limit=` to be explicit.

**Running the equivalent recipe instead?** Do not start a client there.
ESMValTool manages dask itself, and it defaults to the threaded
scheduler; switch it on in your ESMValTool configuration:

```yaml
dask:
  use: local_distributed
```

In [ ]:
import logging
import os

from dask.distributed import Client

# distributed logs every worker registration and teardown at INFO,
# which is hundreds of lines per notebook and buries everything the
# diagnostic prints. Warnings and errors still come through.
for _name in ("distributed", "distributed.scheduler", "distributed.nanny",
              "distributed.worker", "distributed.core",
              "distributed.batched", "distributed.utils_perf"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# Client.current() reuses a cluster if this cell is re-run, instead of
# quietly starting a second one alongside the first.
try:
    worker_count = max(1, int(os.environ.get("PBS_NCPUS", "1")))
except ValueError:
    worker_count = 1
    print("PBS_NCPUS is not an integer; using one worker")
try:
    client = Client.current()
except ValueError:
    try:
        # Separate worker processes: real memory isolation, and what
        # you want on ARE or any batch node.
        client = Client(n_workers=worker_count, threads_per_worker=1)
    except (RuntimeError, OSError) as error:
        # Some containers and locked-down environments cannot spawn
        # worker subprocesses. In-process workers still give a
        # distributed scheduler and a dashboard, just no isolation.
        print(f'process workers unavailable ({error}); '
              f'using in-process workers instead')
        client = Client(n_workers=worker_count, threads_per_worker=1,
                        processes=False)

active_workers = len(client.scheduler_info()['workers'])
print('dashboard:', client.dashboard_link)
print(f'workers: {active_workers} active (requested {worker_count})')
client

## Load the data

In [ ]:
common = dict(project='CMIP6', mip='Omon', exp='historical',
              ensemble='r1i1p1f1', grid='gn',
              timerange='2000/2009')

template = Dataset(short_name='tos', dataset='ACCESS-CM2',
                   **common)

def southern_ocean(dataset):
    """South of 60S on a 1x1 grid, keeping the time axis."""
    cube = dataset.load()
    cube = extract_region(cube, start_longitude=0.,
                          end_longitude=360.,
                          start_latitude=-90.,
                          end_latitude=-60.)
    return regrid(cube, target_grid='1x1', scheme='linear')

fields = {}
for short_name in ['tos', 'sos', 'hfds', 'wfo', 'sfdsi']:
    try:
        fields[short_name] = southern_ocean(
            template.copy(short_name=short_name))
    except Exception as error:
        print(f'{short_name} not available:', error)

## Helpers

In [ ]:
# The helpers below are lifted verbatim from
# cosima_cmip/cosima_common.py by notebooks/build_notebooks.py, so
# this notebook runs exactly the same code as the ESMValTool
# diagnostic script. Edit them there, then rebuild.
import contextlib
import warnings

try:  # TEOS-10, needed by the density helpers
    import gsw
except ImportError:
    gsw = None


EARTH_RADIUS = 6371000.0  # m, matches iris.analysis.cartography


CP_OCEAN = 3992.1  # J kg-1 degC-1, TEOS-10 heat capacity used by MOM6


FILL_VALUE_THRESHOLD = 1.0e19


def masked_data(cube, dtype=float):
    """Realised, fully masked data of a cube.

    Always go through this rather than ``cube.core_data()``.  The latter
    returns a *lazy* dask array, on which ``numpy.ma`` operations are
    silently no-ops: the mask is not applied, and fill values flow
    straight into the arithmetic.  This realises the data, applies any
    mask the file carries, and additionally masks non-finite values and
    unconverted fill values, which real CMOR files do contain.
    """
    data = np.ma.masked_invalid(np.ma.asarray(cube.data, dtype=dtype))
    return np.ma.masked_where(
        np.abs(np.ma.filled(data, 0.0)) >= FILL_VALUE_THRESHOLD, data)


def _coord_points(cube, name):
    """1D points for ``name``, taking column/row 0 of a 2D aux coord."""
    coord = cube.coord(name)
    if coord.ndim == 1:
        return coord.points
    points = coord.points
    # On a curvilinear grid latitude varies down a column and longitude
    # along a row; take the slice that actually varies.
    if name == "latitude":
        return points[:, 0]
    return points[0, :]


def lat_1d(cube):
    """Representative 1D latitude, also for 2D (curvilinear) coordinates."""
    return _coord_points(cube, "latitude")


def lon_1d(cube):
    """Representative 1D longitude, also for 2D coordinates."""
    return _coord_points(cube, "longitude")


def lat_2d(cube):
    """2D latitude field, broadcasting a 1D coordinate if needed."""
    coord = cube.coord("latitude")
    if coord.ndim == 2:
        return coord.points
    lon = cube.coord("longitude").points
    return np.broadcast_to(coord.points[:, None], (coord.shape[0], lon.size))


def lon_2d(cube):
    """2D longitude field, broadcasting a 1D coordinate if needed."""
    coord = cube.coord("longitude")
    if coord.ndim == 2:
        return coord.points
    lat = cube.coord("latitude").points
    return np.broadcast_to(coord.points[None, :], (lat.size, coord.shape[0]))


def guess_bounds(cube, coords=("latitude", "longitude")):
    """Add bounds in place where they are missing."""
    for name in coords:
        if cube.coords(name) and not cube.coord(name).has_bounds():
            try:
                cube.coord(name).guess_bounds()
            except ValueError:  # scalar or single-point coordinate
                pass
    return cube


def horizontal_slice(cube):
    """First slice of a cube over its two horizontal dimensions.

    ``cube.slices(["latitude", "longitude"])`` cannot be used here: on a
    curvilinear grid the two coordinates share both dimensions and iris
    rejects them as non-orthogonal.  Slicing by dimension index works
    for regular and curvilinear grids alike.
    """
    horizontal_dims = sorted(
        set(cube.coord_dims("latitude")) | set(cube.coord_dims("longitude")))
    if cube.ndim == len(horizontal_dims):
        return cube
    index = tuple(slice(None) if dim in horizontal_dims else 0
                  for dim in range(cube.ndim))
    return cube[index]


def spherical_polygon_area(lon_vertices, lat_vertices, radius=EARTH_RADIUS):
    """Area in m2 of spherical polygons given their corner coordinates.

    ``lon_vertices`` and ``lat_vertices`` are in degrees with the corners
    along the last axis, which is the ``(..., 4)`` layout CMOR uses for
    ``vertices_longitude`` / ``vertices_latitude``.  Any number of
    corners works as long as they are ordered around the cell.

    The area is the spherical excess, accumulated over a triangle fan
    from the first corner, using the Van Oosterom & Strackee (1983)
    form:

        tan(E/2) = |v1 . (v2 x v3)|
                   / (1 + v1.v2 + v2.v3 + v3.v1)

    with ``v`` the unit vectors of the corners.  This is *exact* for
    great-circle edges and therefore invariant under rotation of the
    sphere, which matters because a tripolar grid is exactly a rotated
    and stretched one: a formula that is only correct for cells aligned
    with latitude circles would degrade precisely where the grid stops
    being aligned.

    The obvious alternative, the spherical shoelace formula
    ``R^2/2 sum (dlon)(sin lat_i + sin lat_{i+1})``, treats each edge as
    linear in ``(lon, sin lat)`` rather than as a great circle.  It is
    exact for a latitude-longitude box and accurate to second order for
    small cells, but it is not rotation invariant and it fails badly for
    cells spanning large longitude ranges, such as those beside a
    displaced pole.

    The two disagree by about 2e-4 relative on a 2-degree cell, and
    neither is "wrong": they assume different cell edges.  A cell whose
    north and south edges follow *parallels* has the shoelace area,
    which is what ``iris.analysis.cartography.area_weights`` returns; a
    cell with great-circle edges has this one.  Both tile the sphere
    exactly.  The discrepancy is far smaller than the difference
    between either and the model's own ``areacello``, which is why
    ``areacello`` is still preferred when it is published.
    """
    lon = np.deg2rad(np.asarray(lon_vertices, dtype=float))
    lat = np.deg2rad(np.asarray(lat_vertices, dtype=float))
    unit = np.stack([np.cos(lat) * np.cos(lon),
                     np.cos(lat) * np.sin(lon),
                     np.sin(lat)], axis=-1)

    first = unit[..., 0, :]
    excess = np.zeros(unit.shape[:-2])
    for corner in range(1, unit.shape[-2] - 1):
        second = unit[..., corner, :]
        third = unit[..., corner + 1, :]
        triple = np.abs(np.einsum("...i,...i->...",
                                  first, np.cross(second, third)))
        denominator = (1.0
                       + np.einsum("...i,...i->...", first, second)
                       + np.einsum("...i,...i->...", second, third)
                       + np.einsum("...i,...i->...", third, first))
        excess += 2.0 * np.arctan2(triple, denominator)
    return excess * radius**2


def area_from_bounds(cube, radius=EARTH_RADIUS):
    """Cell area in m2 from 2D coordinate bounds, or ``None``.

    This is what makes a tripolar grid workable without ``areacello``.
    CMOR requires bounds on ``latitude`` and ``longitude``, and on a
    curvilinear grid those bounds are the four corner vertices of every
    cell, so the area follows exactly from
    :func:`spherical_polygon_area`.

    ESMValCore does not do this: ``try_adding_calculated_cell_area``
    handles regular and rotated-pole grids and raises
    ``CoordinateMultiDimError`` for everything else.  The information is
    in the file regardless.

    Returns ``None`` when the cube has no usable 2D bounds, so callers
    can fall back.
    """
    horizontal = horizontal_slice(cube)
    lat_coord = horizontal.coord("latitude")
    lon_coord = horizontal.coord("longitude")
    if lat_coord.ndim != 2 or lon_coord.ndim != 2:
        return None
    if lat_coord.bounds is None or lon_coord.bounds is None:
        return None
    if lat_coord.bounds.shape[-1] < 3:
        return None
    return np.ma.masked_invalid(
        spherical_polygon_area(lon_coord.bounds, lat_coord.bounds,
                               radius=radius))


def cell_area(cube):
    """Cell area in m2, preferring the model's own ``areacello``.

    Order of preference:

    1. an ``areacello`` cell measure attached to the cube, which is what
       ESMValCore builds from a ``supplementary_variables`` entry.  This
       is the model's own area and is always the best answer: it
       accounts for the actual grid generation, including any partial
       cells at the coast.
    2. a spherical polygon area from the 2D corner vertices, via
       :func:`area_from_bounds`.  CMOR mandates bounds on latitude and
       longitude, so this works on a tripolar grid with no
       ``areacello`` at all.  It assumes great-circle cell edges on a
       sphere, so it differs from the model's own areas by well under a
       percent for an ocean grid.
    3. spherical areas from 1D latitude/longitude bounds, for a regular
       grid.

    Only if all three fail does this raise, and then with instructions.
    """
    horizontal = horizontal_slice(cube)

    for measure in horizontal.cell_measures():
        if measure.var_name in ("areacello", "areacella", "areacell"):
            return np.ma.masked_invalid(
                np.ma.asarray(
                    horizontal.cell_measure(measure.name()).core_data(),
                    dtype=float).reshape(horizontal.shape))

    from_bounds = area_from_bounds(horizontal)
    if from_bounds is not None:
        return from_bounds

    if horizontal.coord("latitude").ndim > 1:
        raise ValueError(
            "this cube has 2D (curvilinear) latitude/longitude and no cell "
            "corner bounds, so no cell area can be obtained. Either attach "
            "areacello -- in a recipe with\n"
            "    supplementary_variables:\n"
            "      - short_name: areacello\n"
            "        mip: Ofx\n"
            "(add 'exp: piControl' if the experiment you want does not "
            "publish it), in a notebook with "
            "dataset.add_supplementary(short_name='areacello', mip='Ofx') "
            "-- or keep the coordinate bounds, which CMOR requires and "
            "which are enough to compute the area exactly.")

    # iris defaults to an Earth radius of 6367470 m where the cube has
    # no coordinate system, while cell_lengths uses the mean radius of
    # 6371000 m. That is a 0.11% difference in area, small but free to
    # avoid: pin the same sphere so lengths and areas agree.
    horizontal = horizontal.copy()
    guess_bounds(horizontal)
    if horizontal.coord("latitude").coord_system is None:
        sphere = iris.coord_systems.GeogCS(EARTH_RADIUS)
        horizontal.coord("latitude").coord_system = sphere
        horizontal.coord("longitude").coord_system = sphere
    return iris.analysis.cartography.area_weights(horizontal)


def bin_centres(bins):
    """Midpoints of a monotonic array of bin edges."""
    bins = np.asarray(bins, dtype=float)
    return 0.5 * (bins[1:] + bins[:-1])


def bin_sum(values, bin_variable, bins, axis=None):
    """Sum ``values`` into ``bins`` of ``bin_variable``.

    ``values`` and ``bin_variable`` must have the same shape.  Returns
    an array whose first axis runs over the bin centres and whose
    remaining axes are the axes of ``values`` *not* listed in ``axis``.
    With ``axis=None`` everything is collapsed into a 1D profile.

    This is ``xhistogram.xarray.histogram(bin_variable, bins=bins,
    weights=values, dim=axis)`` without the xarray dependency.
    """
    values = np.ma.masked_invalid(np.ma.asarray(values, dtype=float))
    bin_variable = np.ma.masked_invalid(
        np.ma.asarray(bin_variable, dtype=float))
    bins = np.asarray(bins, dtype=float)
    n_bins = bins.size - 1

    if axis is None:
        axis = tuple(range(values.ndim))
    elif isinstance(axis, int):
        axis = (axis,)
    axis = tuple(a % values.ndim for a in axis)
    keep = tuple(a for a in range(values.ndim) if a not in axis)

    moved = np.ma.transpose(values, keep + axis)
    moved_bin = np.ma.transpose(bin_variable, keep + axis)
    kept_shape = moved.shape[:len(keep)]
    flat = moved.reshape(int(np.prod(kept_shape)) if keep else 1, -1)
    flat_bin = moved_bin.reshape(flat.shape)

    out = np.zeros((n_bins,) + kept_shape)
    flat_out = out.reshape(n_bins, -1)
    # np.digitize returns 0 below the first edge and n_bins+1 above the
    # last, so a bin index of 1..n_bins is inside the range.
    for column in range(flat.shape[0]):
        data = flat[column]
        coordinate = flat_bin[column]
        valid = ~np.ma.getmaskarray(data) & ~np.ma.getmaskarray(coordinate)
        if not valid.any():
            continue
        index = np.digitize(
            np.ma.filled(coordinate, np.nan)[valid], bins)
        inside = (index >= 1) & (index <= n_bins)
        flat_out[:, column] = np.bincount(
            index[inside] - 1,
            weights=np.ma.filled(data, 0.0)[valid][inside],
            minlength=n_bins)
    return out


def time_interval_days(cube, use_bounds=True):
    """Length of each time step in days, in the cube's own calendar.

    Do not convert a time coordinate to a hard-coded reference unit to
    do this.  ``cf_units`` refuses to convert between calendars, so
    ``coord.units.convert(points, "days since 1850-01-01")`` raises on
    any cube whose calendar is not ``standard`` -- and CMIP6 is full of
    them: ACCESS-CM2 is ``proleptic_gregorian``, UKESM1 and HadGEM3 are
    ``360_day``, several others are ``noleap``.

    Converting is also the wrong idea even where it works.  Month
    lengths are calendar-dependent, and they are exactly what a
    flux-weighted average needs: every month is 30 days in a
    ``360_day`` model and February never has 29 in a ``noleap`` one.
    Going through ``num2date`` lets the model's own calendar answer.

    Returns per-step lengths from the coordinate bounds, or the spacing
    between points where there are no bounds.
    """
    coord = cube.coord("time")
    if use_bounds and coord.has_bounds():
        dates = coord.units.num2date(coord.bounds)
        return np.array([(end - start).total_seconds() / 86400.0
                         for start, end in dates])

    dates = np.atleast_1d(coord.units.num2date(coord.points))
    if dates.size < 2:
        return np.array([30.0])
    spacing = np.array([(dates[i + 1] - dates[i]).total_seconds() / 86400.0
                        for i in range(dates.size - 1)])
    return np.concatenate([spacing, spacing[-1:]])


@contextlib.contextmanager
def quiet_gsw():
    """Silence gsw's masked-array warning for the enclosed call.

    ``gsw`` emits ``'where' used without 'out', expect uninitialized
    memory in output`` every time it is handed a masked array, which is
    every time here.  The uninitialised values sit only underneath the
    mask and never reach the result -- asserted in
    ``validation/test_cosima_common.py`` -- but the warning fires once
    per call and buries everything else a notebook prints.

    Scoped to this one message so genuine gsw warnings still surface.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", message=".*'where' used without 'out'.*",
            category=UserWarning)
        yield


def _require_gsw():
    if gsw is None:
        raise ImportError(
            "gsw (TEOS-10) is required for the density diagnostics; "
            "install it with `pixi add gsw`")


def absolute_salinity(practical_salinity, pressure, lon, lat):
    """Practical salinity (PSU) to absolute salinity (g/kg)."""
    _require_gsw()
    with quiet_gsw():
        return gsw.SA_from_SP(practical_salinity, pressure, lon, lat)


def conservative_temperature(abs_salinity, potential_temperature):
    """Potential temperature (degC) to conservative temperature (degC)."""
    _require_gsw()
    with quiet_gsw():
        return gsw.CT_from_pt(abs_salinity, potential_temperature)


def thermal_expansion(abs_sal, cons_temp, pressure):
    """Thermal expansion coefficient alpha in K-1."""
    _require_gsw()
    with quiet_gsw():
        return gsw.alpha(abs_sal, cons_temp, pressure)


def haline_contraction(abs_sal, cons_temp, pressure):
    """Haline contraction coefficient beta in kg/g."""
    _require_gsw()
    with quiet_gsw():
        return gsw.beta(abs_sal, cons_temp, pressure)

### Calendars, and why month lengths cannot be borrowed

The transformation is accumulated month by month and weighted by
the length of each month, so those lengths have to be right.

The obvious way to get them — convert the time coordinate to a
fixed reference unit and difference it — fails on CMIP6 data,
twice over.

First it simply raises. `cf_units` refuses to convert between
calendars, and `'days since 1850-01-01'` implies the `standard`
calendar, so any cube on another one stops with
`Unable to convert from ... 'proleptic_gregorian' to ...
'standard'`. ACCESS-CM2 is `proleptic_gregorian`.

Second, and worse where it *does* work: month lengths are a
property of the calendar. UKESM1 and HadGEM3 run on `360_day`,
where every month is 30 days and a year is 360; several models
use `noleap`, where February is never 29. Forcing those onto a
standard calendar would silently mis-weight every month.

`time_interval_days` goes through `num2date` instead, so the
model's own calendar answers. The cell below prints which
calendar it found and the month lengths it derived — a year of
360 days is not a bug, it is UKESM1.

The COSIMA original does not face this: it hard-codes
`months_standard_noleap` because it knows its own model. A
recipe meant to run across CMIP6 cannot assume.

### A word on the density bins

The default bins span $\sigma_1$ = 31.0 to 33.5, which is the
Southern Ocean range the original notebook chose. Cells whose
surface density falls outside that range contribute **nothing**,
silently — they are simply not in any bin.

That is fine while the region is south of 60S, and quietly
misleading if you widen it: a global domain puts most of the
surface ocean outside the range and you get a partial answer that
still looks like a complete one. The cell below reports the
fraction that actually landed in a bin, so widen `SIGMA1_BINS`
(or narrow the region) if it is not close to 1.

## Transformation rates

$\Omega_{\text{heat}} = \alpha Q / c_p$ and
$\Omega_{\text{salt}} = \beta (S_A\,\mathrm{wfo} -
1000\,\mathrm{sfdsi})$, accumulated month by month.

In [ ]:
SIGMA1_BINS = np.arange(31.0, 33.5, 0.02)

tos, sos = fields['tos'], fields['sos']
salinity = masked_data(sos)
temperature = masked_data(tos)

lat = np.broadcast_to(lat_2d(sos)[None, ...], salinity.shape)
lon = np.broadcast_to(lon_2d(sos)[None, ...], salinity.shape)
pressure = np.zeros_like(salinity)      # surface fields

SA = absolute_salinity(salinity, pressure, lon, lat)
CT = conservative_temperature(SA, temperature)

alpha = thermal_expansion(SA, CT, pressure)
beta = haline_contraction(SA, CT, pressure)
with quiet_gsw():
    sigma1 = np.ma.masked_invalid(gsw.sigma1(SA, CT))

# Month lengths in the model's OWN calendar -- see the note
# above. Never convert to a hard-coded reference unit here.
days = time_interval_days(tos)
days_field = days[:, None, None]
print(f"calendar: {tos.coord('time').units.calendar}"
      f'  | month lengths {days.min():.0f}-{days.max():.0f} d'
      f'  | year = {days[:12].sum():.0f} d')

salt_from_water = SA * masked_data(fields['wfo'])
if 'sfdsi' in fields:
    salt_flux = masked_data(fields['sfdsi']) * 1000.0
else:
    print('no sfdsi: the salt term omits the sea-ice salt flux')
    salt_flux = np.ma.zeros(salt_from_water.shape)

heat = alpha * masked_data(fields['hfds']) * days_field
salt = beta * (salt_from_water - salt_flux) * days_field

inside = (sigma1 >= SIGMA1_BINS[0]) & (sigma1 <= SIGMA1_BINS[-1])
print(f'sigma1 spans {np.ma.min(sigma1):.2f} to '
      f'{np.ma.max(sigma1):.2f}; '
      f'{100*np.ma.mean(inside.astype(float)):.0f}% of cells '
      f'fall inside the bin range')

## Bin by surface density

Normalise by the total number of days and the bin width, then
flip the sign so that positive means conversion towards **denser**
water.

In [ ]:
heat_binned = bin_sum(heat, sigma1, SIGMA1_BINS, axis=0)
salt_binned = bin_sum(salt, sigma1, SIGMA1_BINS, axis=0)

total_days = float(np.sum(days))
widths = np.diff(SIGMA1_BINS)[:, None, None]

heat_binned = -heat_binned / (CP_OCEAN * total_days) / widths
salt_binned = -salt_binned / total_days / widths

area = cell_area(tos)
to_sv = lambda binned: np.ma.sum(np.ma.sum(
    binned * np.broadcast_to(area, binned.shape),
    axis=-1), axis=-1) / 1.0e6

swmt_heat = to_sv(np.ma.masked_equal(heat_binned, 0.0))
swmt_salt = to_sv(np.ma.masked_equal(salt_binned, 0.0))
swmt_net = swmt_heat + swmt_salt
centres = bin_centres(SIGMA1_BINS)

## Plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.plot(swmt_net, centres, color='k', label='net')
ax.plot(swmt_heat, centres, color='r', label='heat')
ax.plot(swmt_salt, centres, color='b', label='salt')

ax.axvline(0, color='k', linewidth=0.5)
ax.set_ylim(centres.max(), centres.min())
ax.set_ylabel(r'$\sigma_1$ (kg m$^{-3}$)', fontsize=13)
ax.set_xlabel('Surface water-mass transformation (Sv)',
              fontsize=13)
ax.set_title('Positive = densification')
ax.legend(loc='lower left', frameon=False)
ax.grid(alpha=0.3)
plt.show()

Two things to read off the figure. Positive values are water
becoming denser; negative, lighter. And the *slope*
$\partial_\sigma\Omega$ gives convergence or divergence in
density space: water on the dense side of a positive peak is both
densifying and converging, which is subduction, and the water
feeding it is the divergent band just lighter than the peak.

## Shut the cluster down

Workers hold their memory until the cluster closes. Leaving one running
in an idle kernel is the usual reason a later notebook cannot get the
memory it asks for.

In [ ]:
client.close()